# Training on Colab

Runs `src/train.py` on a Colab GPU and writes results to Google Drive. The published
runs used fp16 on a Tesla T4 and an A100.

Getting the code, while the repository is private, either:
- add a GitHub fine-grained token (this repository, Contents: read-only) as a Colab
  secret named `GITHUB_TOKEN` and switch on its "Notebook access" toggle, or
- upload the repository zip to `/content/bangla-nli-lora.zip` (Files panel).

After the install cell, restart the runtime once and continue from the next cell. Run
training through these cells; `train.py` imports `src/data.py` and only finds it when
run as a script.


In [ ]:
import os, subprocess, zipfile
REPO = "github.com/ArmanMahmud003/bangla-nli-lora"
ZIP = "/content/bangla-nli-lora.zip"

if not os.path.isfile("/content/repo/src/train.py"):
    if os.path.isfile(ZIP):
        zipfile.ZipFile(ZIP).extractall("/content/repo")
        print("code taken from", ZIP)
    else:
        token = None
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception as e:
            print(f"no token available: {type(e).__name__}: {e}")
        url = f"https://x-access-token:{token}@{REPO}" if token else f"https://{REPO}"
        r = subprocess.run(["git", "clone", "-q", url, "/content/repo"],
                           capture_output=True, text=True)
        if r.returncode:
            err = r.stderr.replace(token, "***") if token else r.stderr
            raise RuntimeError(f"clone failed ({'with' if token else 'without'} token):\n{err}")
        print("cloned", "with token" if token else "without token")

!pip install -q -r /content/repo/requirements.txt
!pip install -q git+https://github.com/csebuetnlp/normalizer


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/bangla_nli_runs"
!nvidia-smi --query-gpu=name,memory.total --format=csv

Pilot: one seed at 1% for both methods (about 10 minutes on an A100). Test macro-F1
should land around 0.71 to 0.73.

In [ ]:
!cd /content/repo && python src/train.py --out {OUT} --scratch /content/ckpt --fractions 0.01 --seeds 42

Full grid. Finished runs are skipped, so the cell can be re-run after a disconnect.

In [ ]:
!cd /content/repo && python src/train.py --out {OUT} --scratch /content/ckpt

In [ ]:
!cd /content/repo && python src/analyze.py --artifacts {OUT} --figures {OUT}/figures